In [ ]:
# Select Planet
from pathlib import Path

import cv2
import numpy as np
import ipywidgets

from PIL import Image
from numpy.typing import NDArray


def list_dir(folder: str | Path) -> list[Path]:
    """Scan a folder and return all folders inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    return [f for f in folder.iterdir() if f.is_dir()]


def list_files(folder: str | Path, extension: str | None = None) -> list[Path]:
    """Scan a folder and return all files inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    output_files = []
    for f in folder.iterdir():
        if f.is_file():
            if extension is not None:
                if f.suffix != extension:
                    continue
            output_files.append(f)
    return output_files


def generate_z_normal(x_data: NDArray, y_data: NDArray) -> NDArray:
    """Reconstruct a Z axis normal from X and Y."""
    x = x_data.astype(np.float32) / 255.0
    y = y_data.astype(np.float32) / 255.0

    # Step 2: Convert to [-1, 1]
    x_norm = 2.0 * x - 1.0
    y_norm = 2.0 * y - 1.0

    # Step 3: Reconstruct Z
    # Ensure x^2 + y^2 <= 1 to avoid NaNs due to compression artifacts
    xy_sum_sq = x_norm**2 + y_norm**2
    # Clamp to 1.0 to prevent sqrt of negative numbers
    xy_sum_sq = np.minimum(xy_sum_sq, 1.0)
    z_norm = np.sqrt(1.0 - xy_sum_sq)

    # Convert back to 0 → 255 representation
    z = (z_norm + 1.0) / 2.0
    z_data = (z * 255.0).astype(np.int8)
    return z_data


def scansat_shading(
    colour_map: NDArray, normal_map: NDArray, normal_axis: int = 2
) -> NDArray:
    """Replicate the SCANsat shading code to see what it does."""
    hls_colours = cv2.cvtColor(colour_map, cv2.COLOR_RGB2HLS)

    opacity = 0.8

    lumOver = normal_map[:, :, normal_axis].astype(np.float32) / 255.0
    lum = hls_colours[:, :, 1].astype(np.float32) / 255.0

    new_lum = np.where(
        lum > 0.5,
        (opacity * (1 - (1 - (2 * (lumOver - 0.5))) * (1 - lum))) + (1 - opacity) * lum,
        (opacity * (2 * lumOver * lum)) + (1 - opacity) * lum,
    )

    hls_colours[:, :, 1] = (new_lum * 255.0).astype(np.int8)
    return cv2.cvtColor(hls_colours, cv2.COLOR_HLS2RGB)


def detect_bodies(folder: Path) -> dict[str, Path]:
    """Find body information for values within the folder."""
    bodies = {}

    def iter_search(fpath: Path):
        dirs = list_dir(fpath)
        if "Kopernicus" in [d.stem for d in dirs]:
            bodies[fpath.stem.rsplit("_", 1)[1]] = fpath
            return
        for d in dirs:
            iter_search(d)

    iter_search(folder)
    return bodies


sol_dir = r"C:\Users\rweld\Documents\Kerbal Space Program 1\KSP Stock RSS\GameData\Sol-Textures\PluginData"

bodies = {}
for f in list_dir(sol_dir):
    if f.name[0].isdigit():
        if int(f.name.split("_", 1)[0]):
            bodies.update(detect_bodies(f))

select_widget = ipywidgets.Select(
    options=list(bodies.keys()),
    # value=list(bodies.keys())[0],  # Default selected value
    description="Select Body:",
    disabled=False,
)

select_widget

In [ ]:
# Generate Images for Selected Body

body_name = select_widget.value

images = {f.stem: f for f in list_files(bodies[body_name] / Path("Kopernicus"), ".dds")}

# Height Map
height_image = Image.open(images[f"{body_name}_Height"])
height = np.asarray(height_image, copy=True)
print(f"Height Array is size {height.shape}")
print(f"Height Pixel values between {np.min(height)} to {np.max(height)}\n")

# Normal Map
normal_image = Image.open(images[f"{body_name}_Normal"])
normals = np.asarray(normal_image, copy=True)
normals[:, :, 2] = generate_z_normal(normals[:, :, 0], normals[:, :, 1])
print(f"Normal Array is size {normals.shape}")
print(
    f"Normal Z values between {np.min(normals[:, :, 2])} and {np.max(normals[:, :, 2])}\n"
)

# Colour Map
colour_image = Image.open(images[f"{body_name}_Color"])
colour = np.asarray(colour_image, copy=True)
colour[:, :, 3] = 255  # Make image fully opaque
print(f"Colour Array is size {colour.shape}")

# Apply SCANsat shading
if colour.shape[0] != normals.shape[0]:
    colour = np.asarray(
        Image.fromarray(colour).resize((normals.shape[1], normals.shape[0]))
    )
bumped_colour = scansat_shading(colour, normals, normal_axis=1)

In [ ]:
# Display Coloured Terrain
from jupyter_compare_view import compare


def correct_perspective(save_data: NDArray) -> NDArray:
    """Rotate and shift texture to match in-game map view"""
    save_data = np.rot90(save_data, k=2)
    term = int(save_data.shape[1] / 4)
    save_data = np.concat([save_data[:, term:, :], save_data[:, :term, :]], axis=1)
    # save_data = np.flip(save_data, 0)  # Stock Planets Only
    return save_data


# Rotate and shift to standard map projection
c = correct_perspective(colour)
bc = correct_perspective(bumped_colour)

compare(c, bc, start_mode="horizontal", start_slider_pos=0.5, height=400)